# 01. Clear dataset EDA

Короткий итоговый EDA для clean-датасета `MM`-опционов с правильным `MXI`-фьючерсным underlying.


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project_root = Path('/Users/maria/Desktop/Code/HSE/COURSEBOOK')
data_path = project_root / 'clear/mm_options_with_mxi_underlying_2024_2026.parquet'
df = pd.read_parquet(data_path)
df['TRADEDATE'] = pd.to_datetime(df['TRADEDATE'], errors='coerce')
df['expiry_date'] = pd.to_datetime(df['expiry_date'], errors='coerce')
df.shape


(19574, 34)

## 1. Общая структура


In [2]:
summary = {
    'rows': len(df),
    'unique_option_secids': int(df['SECID'].nunique()),
    'unique_underlying_secids': int(df['UNDERLYING_SECID'].dropna().nunique()),
    'trade_date_min': df['TRADEDATE'].min().date(),
    'trade_date_max': df['TRADEDATE'].max().date(),
    'expiry_min': df['expiry_date'].min().date(),
    'expiry_max': df['expiry_date'].max().date(),
    'option_types': df['option_type'].value_counts(dropna=False).to_dict(),
}
pd.Series(summary)


rows                                          19574
unique_option_secids                             92
unique_underlying_secids                          5
trade_date_min                           2024-01-03
trade_date_max                           2026-05-19
expiry_min                               2026-04-16
expiry_max                               2027-06-17
option_types                {'C': 13217, 'P': 6357}
dtype: object

## 2. Покрытие и контракты underlying


In [3]:
print('Underlying contracts:', sorted(df['UNDERLYING_SHORTNAME'].dropna().unique().tolist()))
display(df['UNDERLYING_SHORTNAME'].value_counts().to_frame('rows'))
display(df.groupby(df['TRADEDATE'].dt.year).size().to_frame('rows'))


Underlying contracts: ['MXI-12.26', 'MXI-3.27', 'MXI-6.26', 'MXI-6.27', 'MXI-9.26']


,rows
UNDERLYING_SHORTNAME,
MXI-6.26,11709
MXI-12.26,2472
MXI-9.26,2337
MXI-3.27,1785
MXI-6.27,1271


,rows
TRADEDATE,
2024,6921
2025,7359
2026,5294


## 3. Страйки, экспирации, пропуски


In [4]:
eda = {
    'unique_strikes': int(df['strike'].nunique()),
    'unique_expiries': int(df['expiry_date'].nunique()),
    'missing_market_price_share': float(df['market_price'].isna().mean()),
    'missing_underlying_price_share': float(df['underlying_price'].isna().mean()),
    'duplicate_secid_tradedate_rows': int(df.duplicated(['SECID', 'TRADEDATE']).sum()),
}
pd.Series(eda)

missing = df[['market_price', 'underlying_price', 'moneyness', 'log_moneyness', 'UNDERLYING_CLOSE', 'UNDERLYING_VOLUME']].isna().mean().sort_values(ascending=False)
missing.to_frame("missing_share")


,missing_share
UNDERLYING_CLOSE,0.274395
UNDERLYING_VOLUME,0.274395
market_price,0.000000
underlying_price,0.000000
moneyness,0.000000
log_moneyness,0.000000


## 4. Очень короткий verdict


In [5]:
verdict = [
    '1. Датасет пересобран с MXI-фьючерсным underlying, а не с индексом IMOEX spot.',
    '2. Покрытие по option rows полное: underlying_price есть для всех строк.',
    '3. Для моделирования базовой цены underlying разумно использовать UNDERLYING_SETTLEPRICE / underlying_price.',
    '4. Поля UNDERLYING_CLOSE и часть trade-полей фьючерса местами пустые, но settlement-based pricing dataset выглядит рабочим.',
    '5. Это более корректная база для следующего шага, чем предыдущий merged dataset с IMOEX spot.',
]
print('\n'.join(verdict))


1. Датасет пересобран с MXI-фьючерсным underlying, а не с индексом IMOEX spot.
2. Покрытие по option rows полное: underlying_price есть для всех строк.
3. Для моделирования базовой цены underlying разумно использовать UNDERLYING_SETTLEPRICE / underlying_price.
4. Поля UNDERLYING_CLOSE и часть trade-полей фьючерса местами пустые, но settlement-based pricing dataset выглядит рабочим.
5. Это более корректная база для следующего шага, чем предыдущий merged dataset с IMOEX spot.
